# 04 — Graph construction

We will produce a `torch_geometric.data.Data` object compatible with
`backend.perception.structural.gat_model.GraphEncoder`.

In [1]:
import numpy as np
import torch
import yfinance as yf
from torch_geometric.data import Data

from backend.config.constants import (
    NUM_SECTORS,
    SECTORS,
    TARGET_TICKERS,
    TICKER_TO_SECTOR,
)

UNIVERSE = sorted(TARGET_TICKERS)
N = len(UNIVERSE)

/home/pyros05/Escritorio/lumina_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pull daily prices

In [2]:
df = yf.download(
    UNIVERSE,
    start="2023-01-01",
    end="2024-01-01",
    interval="1d",
    auto_adjust=False,
    progress=False,
)
close = df["Close"]
returns = np.log(close / close.shift(1)).dropna()

## Node features

32-dim node feature vector composition:
  - sector one-hot:   NUM_SECTORS dims
  - log market cap:   1 dim   (placeholder; yfinance .info is slow)
  - realised vol:     1 dim
  - beta vs SPY:      1 dim
  - reserved zeros:   pad to 32

In [3]:
sector_idx = {s: i for i, s in enumerate(SECTORS)}
spy_ret = returns["SPY"]
features = np.zeros((N, 32), dtype=np.float32)
for i, t in enumerate(UNIVERSE):
    sec = TICKER_TO_SECTOR.get(t, "?")
    if sec in sector_idx:
        features[i, sector_idx[sec]] = 1.0
    features[i, NUM_SECTORS] = 0.0  # market cap placeholder
    features[i, NUM_SECTORS + 1] = float(returns[t].std())
    cov = float(returns[t].cov(spy_ret))
    var_spy = float(spy_ret.var())
    features[i, NUM_SECTORS + 2] = cov / var_spy if var_spy > 0 else 0.0
print("Feature matrix:", features.shape)

Feature matrix: (32, 32)


## Edges

Combine static (sector) + dynamic (correlation > 0.5) edges.

In [4]:
corr = returns.corr().to_numpy()
edges, edge_attrs = [], []
for i in range(N):
    for j in range(N):
        if i == j:
            continue
        same_sector = TICKER_TO_SECTOR.get(UNIVERSE[i]) == TICKER_TO_SECTOR.get(UNIVERSE[j])
        r = float(corr[i, j])
        is_corr_edge = r > 0.5
        if same_sector or is_corr_edge:
            edges.append([i, j])
            # 4-D edge feature: [r, |r|, is_supply_chain=0, weight]
            edge_attrs.append([r, abs(r), 0.0, max(r, 0.5 if same_sector else 0.0)])
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)
print("Edges:", edge_index.shape[1])

Edges: 284


In [5]:
data = Data(
    x=torch.from_numpy(features),
    edge_index=edge_index,
    edge_attr=edge_attr,
)
print(data)

Data(x=[32, 32], edge_index=[2, 284], edge_attr=[284, 4])


## Smoke-run the encoder

In [6]:
from backend.perception.structural.gat_model import GraphEncoder

gnn = GraphEncoder()
z = gnn(data.x, data.edge_index, data.edge_attr)
print("Output:", z.shape)  # (32, DIM_GRAPH)
assert z.shape == (N, 32), z.shape
print("PASS — graph + GATv2 wired correctly.")

Output: torch.Size([32, 32])
PASS — graph + GATv2 wired correctly.
